# DeepLOB on Colab (GPU)

Workflow:
1. Clone code from GitHub
2. Mount Google Drive (data + checkpoints)
3. Download + convert FI-2010 mirror to float32 .npy
4. Install deps
5. Train (Setup 2, k=10)

Code on GitHub; data/weights on Drive. Do NOT commit multi-GB data to git.

In [ ]:
# 1. Clone / update code%cd /content!git clone https://github.com/runchengxie/deeplob-reproduction.git || (cd deeplob-reproduction && git pull)%cd deeplob-reproduction!ls# 2. Mount Drivefrom google.colab import drivedrive.mount('/content/drive')import osDATA_DIR = '/content/drive/MyDrive/DeepLOB/data'CKPT_DIR = '/content/drive/MyDrive/DeepLOB/checkpoints'os.makedirs(DATA_DIR, exist_ok=True)os.makedirs(CKPT_DIR, exist_ok=True)# 3. Download FI-2010 mirror (shanehans/FI2010) CSV, convert to float32 .npy onceimport numpy as npimport pandas as pdnpy_path = os.path.join(DATA_DIR, 'FI2010_normalised.npy')if not os.path.exists(npy_path):    !wget -O fi2010_train.csv https://huggingface.co/datasets/shanehans/FI2010/resolve/main/FI2010_train.csv    df = pd.read_csv('fi2010_train.csv', header=None)    arr = df.to_numpy(dtype=np.float32)    assert arr.shape[1] >= 44, f'expected >=44 cols, got {arr.shape[1]}'    np.save(npy_path, arr)    print('saved', npy_path, arr.shape, arr.dtype)else:    print('reusing', npy_path)# 4. Install deps!pip install -r requirements.txt# 5. Train (Setup 2: 70/15/15 train/val/test split)#    k selects the label column via dataset.K_TO_LABEL_COLUMN.!python src/train.py --dataset fi2010 --data_path "$DATA_DIR/FI2010_normalised.npy" --k 10 --epochs 100 --batch_size 32 --device cuda --checkpoint_dir "$CKPT_DIR"